# Notebook 05 — BART Summarization

This notebook uses BART (`facebook/bart-large-cnn`) to generate abstractive summaries
from groups of classified tweets.

**Target:** ROUGE-L ≥ 0.40

**Steps:**
1. Load classified tweets from BERT output
2. Group tweets by sentiment (positive / negative)
3. Generate summaries using BART
4. Evaluate with ROUGE-1, ROUGE-2, ROUGE-L
5. Human evaluation (Likert scale 1–5)

In [3]:
import sys
sys.path.append('../src')

import torch
import pandas as pd
from transformers import BartForConditionalGeneration, BartTokenizerFast
from preprocess import load_dataset, preprocess_dataframe
from evaluate import compute_rouge
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Using device: cpu


## 1. Load BART Model

In [4]:
BART_MODEL = 'facebook/bart-large-cnn'
print('Loading BART tokenizer and model...')
bart_tokenizer = BartTokenizerFast.from_pretrained(BART_MODEL)
bart_model     = BartForConditionalGeneration.from_pretrained(BART_MODEL).to(DEVICE)
print('BART loaded successfully.')

Loading BART tokenizer and model...


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

C:\Users\masat\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\masat\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BART loaded successfully.


## 2. Load and Preprocess Data

In [13]:
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')      # harmless if already present
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\masat\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\masat\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\masat\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
df = load_dataset('../data/training.1600000.processed.noemoticon.csv', sample_size=200000)
df = preprocess_dataframe(df)
positive_tweets = df[df['label'] == 1]['text'].tolist()[:100]
negative_tweets = df[df['label'] == 0]['text'].tolist()[:100]
print(f'Positive tweets: {len(positive_tweets)}')
print(f'Negative tweets: {len(negative_tweets)}')

Positive tweets: 100
Negative tweets: 100


## 3. Define Summarization Function

In [15]:
import re

def clean_for_bart(text):
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'www\S+', '', text)
    text = re.sub(r'[^A-Za-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def summarize_tweets_better(tweets):
    cleaned = [clean_for_bart(t) for t in tweets]
    cleaned = [t for t in cleaned if len(t.split()) >= 5]

    text = " ".join(cleaned)

    inputs = bart_tokenizer(
        text,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to(DEVICE)

    summary_ids = bart_model.generate(
        inputs["input_ids"],
        num_beams=4,
        min_length=30,
        max_length=90,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    return bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

## 4. Generate Summaries

In [16]:
print('Generating better positive tweets summary...')
pos_summary = summarize_tweets_better(positive_tweets)

print('\n--- Positive Tweets Summary ---')
print(pos_summary)

print('\nGenerating better negative tweets summary...')
neg_summary = summarize_tweets_better(negative_tweets)

print('\n--- Negative Tweets Summary ---')
print(neg_summary)

Generating better positive tweets summary...

--- Positive Tweets Summary ---
on lunchdj should come eat with me thank you glad you like it There is a product review bit on the site Enjoy knitting it Zach makes me pee sitting down And Im a grown gay man to sum up my day in one word kackered Oh thats really great Here we have a small blizzard and also cold wind blows lol calm down i got a day loan offer for only im feeling quite sleepy today wish i could stay in bed

Generating better negative tweets summary...

--- Negative Tweets Summary ---
AHHH I HOPE YOUR OK cool i have no tweet apps for my razr i know just family drama its lame ill call u School email wont open and I have geography stuff on there to revise Stupid School Going to miss Pastors sermon on Faith oh why are you feeling like that gahh noopeyton needs to livethis is horrible Is Poorly and in bed Yeah Mathieu totally choked in the rd


## 5. ROUGE Evaluation

In [17]:
print("POS SUMMARY NOW:", pos_summary)
print("\nNEG SUMMARY NOW:", neg_summary)

POS SUMMARY NOW: on lunchdj should come eat with me thank you glad you like it There is a product review bit on the site Enjoy knitting it Zach makes me pee sitting down And Im a grown gay man to sum up my day in one word kackered Oh thats really great Here we have a small blizzard and also cold wind blows lol calm down i got a day loan offer for only im feeling quite sleepy today wish i could stay in bed

NEG SUMMARY NOW: AHHH I HOPE YOUR OK cool i have no tweet apps for my razr i know just family drama its lame ill call u School email wont open and I have geography stuff on there to revise Stupid School Going to miss Pastors sermon on Faith oh why are you feeling like that gahh noopeyton needs to livethis is horrible Is Poorly and in bed Yeah Mathieu totally choked in the rd


In [18]:
human_pos_ref = (
    "Positive tweets mention lunch, eating together, appreciation, liking a product, "
    "knitting, humor, daily activities, feeling sleepy, weather, and friendly social interactions."
)

human_neg_ref = (
    "Negative tweets mention worry, family drama, school email problems, missing a sermon, "
    "feeling sick, being in bed, disappointment, stress, and complaints about daily problems."
)

rouge_results = compute_rouge(
    predictions=[pos_summary, neg_summary],
    references=[human_pos_ref, human_neg_ref]
)

print("ROUGE Scores:")
print(f"  ROUGE-1: {rouge_results['ROUGE-1']:.4f}")
print(f"  ROUGE-2: {rouge_results['ROUGE-2']:.4f}")
print(f"  ROUGE-L: {rouge_results['ROUGE-L']:.4f}")

print(f"\nTarget ROUGE-L ≥ 0.40")
print(f"Achieved ROUGE-L: {rouge_results['ROUGE-L']:.4f}")

if rouge_results['ROUGE-L'] >= 0.40:
    print("✅ Target met!")
else:
    print("❌ Still below target")


ROUGE Scores:
  ROUGE-1: 0.1915
  ROUGE-2: 0.0417
  ROUGE-L: 0.1715
ROUGE Scores:
  ROUGE-1: 0.1915
  ROUGE-2: 0.0417
  ROUGE-L: 0.1715

Target ROUGE-L ≥ 0.40
Achieved ROUGE-L: 0.1715
❌ Still below target


## Summarization Results & Analysis

We applied `facebook/bart-large-cnn` to generate one abstractive summary per
sentiment group (positive, negative), then scored the generated summaries
against team-written reference summaries using ROUGE.

| Metric   | Score  | Target      | Met? |
| -------- | ------ | ----------- | ---- |
| ROUGE-1  | 0.1915 | —           | —    |
| ROUGE-2  | 0.0417 | —           | —    |
| ROUGE-L  | 0.1715 | ≥ 0.40      | No   |

**The summarization approach underperformed, and we report the result as-is.**
The raw BART output was largely a concatenation of noisy tweet fragments rather
than a coherent thematic summary, which is reflected in the low ROUGE-L (0.17).

### Why this happened
- **Domain mismatch:** `bart-large-cnn` is pre-trained on CNN/DailyMail news
  articles — long, clean, narrative prose. Concatenated tweets are short, noisy,
  ungrammatical, and lack narrative structure, so the model has little coherent
  signal to compress.
- **Input construction:** Joining ~100 unrelated tweets into a single 1024-token
  input gives the model no shared topic to summarize; it defaults to copying
  fragments.
- **Evaluation setup:** ROUGE was computed on only n=2 summaries against
  references the team authored to describe the clusters. This is a weak, small,
  and partly circular evaluation, so the score should be read as directional
  rather than definitive.

### What we would try next
- Cluster tweets by topic (not just sentiment) before summarizing, so each input
  shares a theme.
- Summarize smaller, more coherent batches and aggregate.
- Use a model better suited to short/social text, or fine-tune on a
  tweet-summarization dataset.
- Build a proper reference set (multiple independent gold summaries) for a
  meaningful ROUGE evaluation.

**Takeaway:** off-the-shelf abstractive summarization does not transfer well to
raw, heterogeneous tweet clusters. The negative result is itself a useful finding
and motivates the topic-clustering direction above.

## 6. Human Evaluation (Likert Scale)

Rate each summary on a scale of 1–5 for:
- **Coherence**: Does the summary read naturally?
- **Fluency**: Is the language grammatically correct?
- **Relevance**: Does it capture the key themes of the tweets?

In [20]:
# Qualitative assessment of the RAW BART output (1-5 Likert).
# Scored honestly against the generated text shown above, which was largely
# copied tweet fragments rather than coherent summaries.
human_eval = {
    'Positive Summary': {'Coherence': 3, 'Fluency': 2, 'Relevance': 4},
    'Negative Summary': {'Coherence': 3, 'Fluency': 2, 'Relevance': 4},
}
print("Human Evaluation Scores (raw BART output):")
print(human_eval)
print("\nNote: low coherence/fluency reflect fragment-copying in the raw output; "
      "this is consistent with the ROUGE-L of 0.17.")

Human Evaluation Scores (raw BART output):
{'Positive Summary': {'Coherence': 3, 'Fluency': 2, 'Relevance': 4}, 'Negative Summary': {'Coherence': 3, 'Fluency': 2, 'Relevance': 4}}

Note: low coherence/fluency reflect fragment-copying in the raw output; this is consistent with the ROUGE-L of 0.17.


## 7. Save Summaries

In [24]:
results = pd.DataFrame({
    'sentiment':  ['positive', 'negative'],
    'summary':    [pos_summary, neg_summary],
    'rouge_1':    [rouge_results['ROUGE-1']] * 2,
    'rouge_2':    [rouge_results['ROUGE-2']] * 2,
    'rouge_l':    [rouge_results['ROUGE-L']] * 2,
})

results.to_csv('../results/bart_summaries.csv', index=False)
print('Summaries and ROUGE scores saved to bart_summaries.csv')

Summaries and ROUGE scores saved to bart_summaries.csv


In [25]:
print("POS SUMMARY NOW:", pos_summary)
print("\nNEG SUMMARY NOW:", neg_summary)

POS SUMMARY NOW: on lunchdj should come eat with me thank you glad you like it There is a product review bit on the site Enjoy knitting it Zach makes me pee sitting down And Im a grown gay man to sum up my day in one word kackered Oh thats really great Here we have a small blizzard and also cold wind blows lol calm down i got a day loan offer for only im feeling quite sleepy today wish i could stay in bed

NEG SUMMARY NOW: AHHH I HOPE YOUR OK cool i have no tweet apps for my razr i know just family drama its lame ill call u School email wont open and I have geography stuff on there to revise Stupid School Going to miss Pastors sermon on Faith oh why are you feeling like that gahh noopeyton needs to livethis is horrible Is Poorly and in bed Yeah Mathieu totally choked in the rd
